In [6]:
import re, os

In [7]:
TWO_WORDS = ["Korea", "Union", "Republic", "States", "Salvador", "Arabia", "Ireland", "Zealand", "Rica", "Herzegovina", "Macedonia", "Tobago", "Leone", "d'Ivoire", "Coast", "Verde", 'Montenegro']
def get_team_1_and_2(line):
    if not re.match(r".*\s\d+.\d+\s.*", line) or "@" not in line:
        return
    try:
        score = re.search(r"\s\d+-\d+\s", line).group().strip()
    except:
        return
    team_1 = line.split(score)[0].strip().split()[-1]
    if team_1 in TWO_WORDS:
        try:
            first_word = line.split(score)[0].strip().split()[-2]
            if (team_1 == "Ireland" and first_word in ["Republic", "Northern"]) or team_1 != "Ireland":
                team_1 = first_word + " " + team_1
        except:
            pass
    team_2 = line.split("@")[0].strip().split()[-1]
    if team_2 in TWO_WORDS:
        try:
            first_word = line.split("@")[0].strip().split()[-2]
            if (team_2 == "Ireland" and first_word in ["Republic", "Northern"]) or team_2 != "Ireland":
                team_2 = first_word + " " + team_2
        except:
            pass
    team_1_score, team_2_score = score.split("-")
    return team_1, team_2, team_1_score, team_2_score

In [8]:
data = {}
for wc in os.listdir("../data/worldcup"):
    if not re.match(r"\d{4}", wc):
            continue
    year, location = wc.split("--")
    if year == "2026": 
        continue
    data[year] = []
    if year != "2014":
        with open(f"../data/worldcup/{wc}/cup.txt", "r") as f:
            for line in f.readlines():
                res = get_team_1_and_2(line)
                if res is None:
                    continue
                data[year].append(
                    {
                        "team_1": res[0],
                        "team_2": res[1],
                        "team_1_score": res[2],
                        "team_2_score": res[3],
                        "phase": "group"
                    }
                )
        if os.path.exists(f"../data/worldcup/{wc}/cup_finals.txt"):
            with open(f"../data/worldcup/{wc}/cup_finals.txt", "r") as f:
                for line in f.readlines():
                    res = get_team_1_and_2(line)
                    if res is None:
                        continue
                    data[year].append(
                        {
                            "team_1": res[0],
                            "team_2": res[1],
                            "team_1_score": res[2],
                            "team_2_score": res[3],
                            "phase": "knockout"
                        }
                    )
    else:
        with open(f"../data/worldcup/{wc}/cup.txt", "r") as f:
            # 16:00 UTC-3  Spain v Netherlands  1-5 (1-1)    @ Arena Fonte Nova, Salvador
            for line in f.readlines():
                try:
                    res = get_team_1_and_2(line)
                    if res is None:
                        continue
                    team_1_score = res[2]
                    team_2_score = res[3]
                    first, second = line.split(" v ")
                    team_1 = first.strip().split()[-1]
                    if team_1 in TWO_WORDS:
                        try:
                            first_word = first.strip().split()[-2]
                            if (team_1 == "Ireland" and first_word in ["Republic", "Northern"]) or team_1 != "Ireland":
                                team_1 = first_word + " " + team_1
                        except:
                            pass
                    team_2 = second.strip().split()[0]
                    if team_2 in TWO_WORDS:
                        try:
                            first_word = second.strip().split()[1]
                            if (team_2 == "Ireland" and first_word in ["Republic", "Northern"]) or team_2 != "Ireland":
                                team_2 = first_word + " " + team_2
                        except:
                            pass
                    data[year].append(
                        {
                            "team_1": team_1,
                            "team_2": team_2,
                            "team_1_score": team_1_score,
                            "team_2_score": team_2_score,
                            "phase": "group"
                        }
                    )
                except:
                    res = get_team_1_and_2(line)
                    if res is None:
                        continue
                    data[year].append(
                        {
                            "team_1": res[0],
                            "team_2": res[1],
                            "team_1_score": res[2],
                            "team_2_score": res[3],
                            "phase": "group"
                        }
                    )
            
        

In [9]:
import json
with open("preprocessed_data/fixtures.json", "w") as f:
    json.dump(data, f, indent=4)

In [10]:
len(data["1998"])

64